## Identify Next Batch

In [0]:
%run ../00-common/01.environment-config

In [0]:
control_table=f"{catalog_name}.{control_schema}.batch_control"

In [0]:
from pyspark.sql import functions as F
#Get batch folders from landing
landing_batches=sorted([
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folder_path)
    if file.isDir()
])
#read tracked batches
if spark.catalog.tableExists(control_table):
    tracked_batches=[
        row.batch_id
        for row in (
            spark.table(control_table)
            .filter(F.col("status").isin("inprogress","completed"))
            .select("batch_id")
            .collect()
        )

    ] 
else:
    tracked_batches=[]

#Identify eraliest unprocessed batch
new_batches=sorted(list(set(landing_batches)-set(tracked_batches)))
next_batches=new_batches[0] if new_batches else None

print(f"Landing batches:{landing_batches}")
print(f"Tracked batches:{tracked_batches}" )
print(f"Next batch to process :{next_batches}" )

if next_batches is None:
    dbutils.jobs.taskValues.set("p_batch_id", "")
    dbutils.jobs.taskValues.set("has_batch", "false")
else:
    dbutils.jobs.taskValues.set("p_batch_id", next_batches)
    dbutils.jobs.taskValues.set("has_batch", "true")







Landing batches:['2025-01']
Tracked batches:[]
Next batch to process :2025-01
